<a href="https://colab.research.google.com/github/arthisri2810-dot/earthquake-streamlit-dashboard/blob/main/earthquake.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install the required packages

In [93]:
!pip install sqlalchemy pymysql requests pandas python-dotenv


Fetch USGS earthquake data (last 5 years)

In [94]:
import requests
import pandas as pd
from datetime import datetime, timezone
import calendar

records = []

def fetch_month(year, month):
    url = "https://earthquake.usgs.gov/fdsnws/event/1/query"

    # Get number of days in the current month (handles 28/29/30/31)
    days = calendar.monthrange(year, month)[1]

    # Build proper date strings
    start = f"{year}-{month:02d}-01"
    end = f"{year}-{month:02d}-{days}"

    params = {
        "format": "geojson",
        "starttime": start,
        "endtime": end,
        "minmagnitude": 2.5
    }

    r = requests.get(url, params=params)
    r.raise_for_status()
    data = r.json()

    for feat in data["features"]:
        prop = feat["properties"]
        geom = feat["geometry"]

        # Safely handle missing geometry
        coords = geom["coordinates"] if geom and "coordinates" in geom else [None, None, None]

        record = {
            "id": feat.get("id"),
            "time": prop.get("time"),
            "updated": prop.get("updated"),
            "latitude": coords[1],
            "longitude": coords[0],
            "depth_km": coords[2],
            "mag": prop.get("mag"),
            "magType": prop.get("magType"),
            "place": prop.get("place"),
            "status": prop.get("status"),
            "tsunami": prop.get("tsunami"),
            "sig": prop.get("sig"),
            "net": prop.get("net"),
            "nst": prop.get("nst"),
            "dmin": prop.get("dmin"),
            "rms": prop.get("rms"),
            "gap": prop.get("gap"),
            "magError": prop.get("magError"),
            "depthError": prop.get("depthError"),
            "magNst": prop.get("magNst"),
            "locationSource": prop.get("locationSource"),
            "magSource": prop.get("magSource"),
            "types": prop.get("types"),
            "ids": prop.get("ids"),
            "sources": prop.get("sources"),
            "type": prop.get("type"),
            "alert": (prop.get("alert") or "").lower(),
        }

        records.append(record)

# Choose last 5 years (NO deprecation warning, works in all Python versions)
end_year = datetime.now(timezone.utc).year
start_year = end_year - 4

# Fetch month-by-month
for year in range(start_year, end_year + 1):
    for month in range(1, 13):
        fetch_month(year, month)

# Display count
len(records)



109414

Create the DataFrame:

In [95]:
df = pd.DataFrame(records)
df.head(), df.shape


(             id           time        updated  latitude  longitude  depth_km  \
 0    us7000ghz5  1643586465730  1650045441040   29.6987    50.3717      10.0   
 1    us7000ghz3  1643586327152  1650045441040   -7.3041   125.7897      10.0   
 2  pr2022030007  1643585370980  1659788660048   18.4700   -66.7005      99.0   
 3    us6000gzy2  1643584367785  1650045421040   52.3399  -169.9284      35.0   
 4    us7000ghyz  1643582951760  1650045441040  -18.2963  -177.9495     555.8   
 
     mag magType                               place    status  ...  magError  \
 0  4.20      mb  19 km NW of Bandar-e Gen?veh, Iran  reviewed  ...      None   
 1  4.60      mb   135 km N of Metinaro, Timor Leste  reviewed  ...      None   
 2  3.47      md      1 km E of Arecibo, Puerto Rico  reviewed  ...      None   
 3  2.90      ml        98 km SW of Nikolski, Alaska  reviewed  ...      None   
 4  4.60      mb            290 km E of Levuka, Fiji  reviewed  ...      None   
 
    depthError magNst  l

Clean data & add derived columns (Python + Regex)

 Convert time fields

In [96]:
df["time"] = pd.to_datetime(df["time"], unit="ms", errors="coerce")
df["updated"] = pd.to_datetime(df["updated"], unit="ms", errors="coerce")


Numeric conversions

In [97]:
numeric_cols = [
    "latitude", "longitude", "depth_km", "mag",
    "tsunami", "sig", "nst", "dmin", "rms", "gap",
    "magError", "depthError", "magNst"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Fill nulls where needed (simple approach)
df[numeric_cols] = df[numeric_cols].fillna(0)
df["tsunami"] = df["tsunami"].fillna(0).astype(int)


Clean text fields + extract country (Regex)

Country from place (very approximate, but ok for this project):

In [98]:
import re

def extract_country(place):
    if not isinstance(place, str):
        return "Unknown"
    # typical place format: "100 km SE of City, Country"
    parts = place.split(",")
    if len(parts) > 1:
        return parts[-1].strip()
    # if no comma, just return the last word as fallback
    words = place.strip().split(" ")
    return words[-1] if words else "Unknown"

df["country"] = df["place"].apply(extract_country)
df["magType"] = df["magType"].fillna("unknown").str.lower()
df["status"] = df["status"].fillna("unknown").str.lower()
df["type"] = df["type"].fillna("unknown").str.lower()
df["net"] = df["net"].fillna("unknown").str.lower()
df["sources"] = df["sources"].fillna("")
df["types"] = df["types"].fillna("")
df["ids"] = df["ids"].fillna("")
df["alert"] = df["alert"].fillna("none")


 Derived columns

In [99]:
df["year"] = df["time"].dt.year
df["month"] = df["time"].dt.month
df["day"] = df["time"].dt.day
df["day_of_week"] = df["time"].dt.day_name()
df["hour"] = df["time"].dt.hour

def depth_category(d):
    if d is None:
        return "unknown"
    if d < 70:
        return "shallow"
    elif d < 300:
        return "intermediate"
    else:
        return "deep"

df["depth_category"] = df["depth_km"].apply(depth_category)

def mag_category(m):
    if m is None:
        return "unknown"
    if m < 4.0:
        return "minor"
    elif m < 5.0:
        return "light"
    elif m < 6.0:
        return "moderate"
    elif m < 7.0:
        return "strong"
    elif m < 8.0:
        return "major"
    else:
        return "great"

df["mag_category"] = df["mag"].apply(mag_category)


casualties & economic loss

In [100]:
df["casualties"] = 0
df["economic_loss"] = 0.0


uploading ca files

In [103]:
from google.colab import files
files.upload()


Saving isrgrootx1 (3).pem to isrgrootx1 (3).pem


{'isrgrootx1 (3).pem': b'-----BEGIN CERTIFICATE-----\nMIIFazCCA1OgAwIBAgIRAIIQz7DSQONZRGPgu2OCiwAwDQYJKoZIhvcNAQELBQAw\nTzELMAkGA1UEBhMCVVMxKTAnBgNVBAoTIEludGVybmV0IFNlY3VyaXR5IFJlc2Vh\ncmNoIEdyb3VwMRUwEwYDVQQDEwxJU1JHIFJvb3QgWDEwHhcNMTUwNjA0MTEwNDM4\nWhcNMzUwNjA0MTEwNDM4WjBPMQswCQYDVQQGEwJVUzEpMCcGA1UEChMgSW50ZXJu\nZXQgU2VjdXJpdHkgUmVzZWFyY2ggR3JvdXAxFTATBgNVBAMTDElTUkcgUm9vdCBY\nMTCCAiIwDQYJKoZIhvcNAQEBBQADggIPADCCAgoCggIBAK3oJHP0FDfzm54rVygc\nh77ct984kIxuPOZXoHj3dcKi/vVqbvYATyjb3miGbESTtrFj/RQSa78f0uoxmyF+\n0TM8ukj13Xnfs7j/EvEhmkvBioZxaUpmZmyPfjxwv60pIgbz5MDmgK7iS4+3mX6U\nA5/TR5d8mUgjU+g4rk8Kb4Mu0UlXjIB0ttov0DiNewNwIRt18jA8+o+u3dpjq+sW\nT8KOEUt+zwvo/7V3LvSye0rgTBIlDHCNAymg4VMk7BPZ7hm/ELNKjD+Jo2FR3qyH\nB5T0Y3HsLuJvW5iB4YlcNHlsdu87kGJ55tukmi8mxdAQ4Q7e2RCOFvu396j3x+UC\nB5iPNgiV5+I3lg02dZ77DnKxHZu8A/lJBdiB3QW0KtZB6awBdpUKD9jf1b0SHzUv\nKBds0pjBqAlkd25HN7rOrFleaJ1/ctaJxQZBKT5ZPt0m9STJEadao0xAH0ahmbWn\nOlFuhjuefXKnEgV4We0+UXgVCwOPjdAvBbI+e0ocS3MFEvzG6uBQE3xDk3SzynTn\njh8BCNAw1FtxNrQHusEwMF

TIDB connection

In [104]:
from sqlalchemy import create_engine, text

host = "gateway01.ap-southeast-1.prod.aws.tidbcloud.com"
port = 4000
user = "31PZhzCNeELY4jm.root"
password = "O8N8mrUbCKtnPi7G"
database = "earthquake"

CA_FILE = "/content/isrgrootx1 (3).pem"   # Change name if different!!

engine = create_engine(
    f"mysql+pymysql://{user}:{password}@{host}:{port}/{database}",
    connect_args={
        "ssl": {
            "ca": CA_FILE
        }
    }
)

try:
    with engine.connect() as conn:
        print(conn.execute(text("SELECT 1")).fetchall())
except Exception as e:
    print("Connection error:", e)


[(1,)]


5. Create earthquakes table in TiDB

In [105]:
create_table_sql = """
CREATE TABLE IF NOT EXISTS earthquake (
    id              VARCHAR(50) PRIMARY KEY,
    time            DATETIME,
    updated         DATETIME,
    latitude        DOUBLE,
    longitude       DOUBLE,
    depth_km        DOUBLE,
    mag             DOUBLE,
    magType         VARCHAR(10),
    place           TEXT,
    status          VARCHAR(20),
    tsunami         TINYINT,
    sig             INT,
    net             VARCHAR(10),
    nst             DOUBLE,
    dmin            DOUBLE,
    rms             DOUBLE,
    gap             DOUBLE,
    magError        DOUBLE,
    depthError      DOUBLE,
    magNst          DOUBLE,
    locationSource  VARCHAR(20),
    magSource       VARCHAR(20),
    types           TEXT,
    ids             TEXT,
    sources         TEXT,
    type            VARCHAR(20),
    alert           VARCHAR(10),
    country         VARCHAR(60),
    year            INT,
    month           INT,
    day             INT,
    day_of_week     VARCHAR(15),
    hour            INT,
    depth_category  VARCHAR(20),
    mag_category    VARCHAR(20),
    casualties      INT,
    economic_loss   DOUBLE
);
"""

with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS earthquakes"))
    conn.execute(text(create_table_sql))
print("Table created.")


Table created.


Remove duplicates

In [106]:
# Remove rows with NULL/empty/invalid IDs
df = df[df["id"].notna()]
df = df[df["id"] != ""]

# Drop exact duplicates
df = df.drop_duplicates(subset=["id"], keep="first")

df.shape


(109414, 37)

Insert DataFrame into TiDB

In [107]:
cols = [
    "id", "time", "updated", "latitude", "longitude", "depth_km",
    "mag", "magType", "place", "status", "tsunami", "sig", "net",
    "nst", "dmin", "rms", "gap", "magError", "depthError", "magNst",
    "locationSource", "magSource", "types", "ids", "sources", "type",
    "alert", "country", "year", "month", "day", "day_of_week", "hour",
    "depth_category", "mag_category", "casualties", "economic_loss"
]

df_to_db = df[cols].copy()

# Insert with to_sql
df_to_db.to_sql(
    "earthquake",
    con=engine,
    if_exists="append",
    index=False,
    method="multi",
    chunksize=1000
)

print("Inserted rows:", len(df_to_db))


Inserted rows: 109414


Load All 30 SQL Queries

In [108]:
queries = {

"1. Top 10 strongest earthquakes":
"""
SELECT * FROM earthquake ORDER BY mag DESC LIMIT 10;
""",

"2. Top 10 deepest earthquake":
"""
SELECT * FROM earthquake ORDER BY depth_km DESC LIMIT 10;
""",

"3. Shallow earthquake < 50km & mag > 7.5":
"""
SELECT * FROM earthquake WHERE depth_km < 50 AND mag > 7.5;
""",

"4. Average depth per country":
"""
SELECT country, AVG(depth_km) AS avg_depth
FROM earthquake GROUP BY country ORDER BY avg_depth DESC;
""",

"5. Average magnitude by magType":
"""
SELECT magType, AVG(mag) AS avg_mag
FROM earthquake GROUP BY magType ORDER BY avg_mag DESC;
""",

"6. Year with most earthquake":
"""
SELECT year, COUNT(*) AS total FROM earthquake
GROUP BY year ORDER BY total DESC LIMIT 1;
""",

"7. Month with highest earthquake":
"""
SELECT year, month, COUNT(*) AS total
FROM earthquake GROUP BY year, month
ORDER BY total DESC LIMIT 1;
""",

"8. Most common day of week":
"""
SELECT day_of_week, COUNT(*) AS total
FROM earthquake GROUP BY day_of_week ORDER BY total DESC;
""",

"9. Earthquakes per hour":
"""
SELECT hour, COUNT(*) AS total
FROM earthquake GROUP BY hour ORDER BY hour;
""",

"10. Most active reporting network":
"""
SELECT net, COUNT(*) AS total
FROM earthquake GROUP BY net ORDER BY total DESC LIMIT 1;
""",

"11. Top 5 locations with highest casualties":
"""
SELECT place, SUM(casualties) AS total
FROM earthquake GROUP BY place
ORDER BY total DESC LIMIT 5;
""",

"12. Total economic loss per country":
"""
SELECT country, SUM(economic_loss) AS total
FROM earthquake GROUP BY country ORDER BY total DESC;
""",

"13. Avg economic loss by alert level":
"""
SELECT alert, AVG(economic_loss) AS avg_loss
FROM earthquake GROUP BY alert ORDER BY avg_loss DESC;
""",

"14. Reviewed vs Automatic":
"""
SELECT status, COUNT(*) AS total
FROM earthquake GROUP BY status;
""",

"15. Earthquake type count":
"""
SELECT type, COUNT(*) AS total
FROM earthquake GROUP BY type ORDER BY total DESC;
""",

"16. Data type counts":
"""
SELECT
SUM(CASE WHEN types LIKE '%dyfi%' THEN 1 END) AS dyfi_count,
SUM(CASE WHEN types LIKE '%shakemap%' THEN 1 END) AS shakemap_count,
COUNT(*) AS total_events
FROM earthquake;
""",

"17. Average RMS and gap per country":
"""
SELECT country, AVG(rms) AS avg_rms, AVG(gap) AS avg_gap
FROM earthquake GROUP BY country ORDER BY avg_rms DESC;
""",

"18. Events with nst > 50":
"""
SELECT * FROM earthquake WHERE nst > 50;
""",

"19. Tsunami events per year":
"""
SELECT year, COUNT(*) AS tsunami_events
FROM earthquake WHERE tsunami=1 GROUP BY year;
""",

"20. Earthquake per alert level":
"""
SELECT alert, COUNT(*) AS total FROM earthquakes GROUP BY alert;
""",

"21. Top 5 countries by avg magnitude (past 10 years)":
"""
SELECT country, AVG(mag) AS avg_mag, COUNT(*) AS events
FROM earthquake
WHERE year >= YEAR(CURDATE()) - 10
GROUP BY country HAVING events >= 10
ORDER BY avg_mag DESC LIMIT 5;
""",

"22. Countries with shallow & deep same month":
"""
SELECT country, year, month
FROM earthquake
GROUP BY country, year, month
HAVING SUM(CASE WHEN depth_category='shallow' THEN 1 END) > 0
AND SUM(CASE WHEN depth_category='deep' THEN 1 END) > 0;
""",

"23. Year-over-year growth":
"""
WITH y AS (
 SELECT year, COUNT(*) total FROM earthquake GROUP BY year
),
g AS (
 SELECT year, total,
 LAG(total) OVER (ORDER BY year) AS prev_total
 FROM y
)
SELECT year, total, prev_total,
((total - prev_total) / prev_total) * 100 AS growth_rate
FROM g;
""",

"24. Top 3 most active countries":
"""
SELECT country, COUNT(*) AS total, AVG(mag) AS avg_mag,
COUNT(*) * AVG(mag) AS score
FROM earthquake GROUP BY country ORDER BY score DESC LIMIT 3;
""",

"25. Avg depth near equator":
"""
SELECT country, AVG(depth_km) AS avg_depth
FROM earthquake WHERE latitude BETWEEN -5 AND 5
GROUP BY country ORDER BY avg_depth DESC;
""",

"26. Shallow to deep ratio":
"""
SELECT country,
SUM(CASE WHEN depth_category='shallow' THEN 1 END) AS shallow,
SUM(CASE WHEN depth_category='deep' THEN 1 END) AS deep,
SUM(CASE WHEN depth_category='shallow' THEN 1 END) /
NULLIF(SUM(CASE WHEN depth_category='deep' THEN 1 END),0) AS ratio
FROM earthquake GROUP BY country ORDER BY ratio DESC;
""",

"27. Magnitude difference (tsunami vs non)":
"""
SELECT
AVG(CASE WHEN tsunami=1 THEN mag END) AS tsunami_mag,
AVG(CASE WHEN tsunami=0 THEN mag END) AS non_tsunami_mag,
AVG(CASE WHEN tsunami=1 THEN mag END) -
AVG(CASE WHEN tsunami=0 THEN mag END) AS diff
FROM earthquake;
""",

"28. Worst reliability (RMS & gap)":
"""
SELECT * FROM earthquake ORDER BY rms DESC, gap DESC LIMIT 50;
""",

"29. Consecutive quakes within 1 hour (same country)":
"""
SELECT e1.id AS eq1, e2.id AS eq2, e1.country,
TIMESTAMPDIFF(MINUTE, e1.time, e2.time) AS minutes_diff
FROM earthquake e1
JOIN earthquake e2
ON e2.time > e1.time
AND TIMESTAMPDIFF(MINUTE, e1.time, e2.time) <= 60
AND e1.country = e2.country
LIMIT 100;
""",

"30. Countries with most deep quakes":
"""
SELECT country, COUNT(*) AS deep_count
FROM earthquake WHERE depth_km > 300
GROUP BY country ORDER BY deep_count DESC LIMIT 10;
"""
}


displaying one data frame

In [109]:
import pandas as pd

def run_query(qname):
    sql = queries[qname]
    with engine.connect() as conn:
        return pd.read_sql(sql, conn)

run_query("1. Top 10 strongest earthquakes")


,id,time,updated,latitude,longitude,depth_km,mag,magType,place,status,...,country,year,month,day,day_of_week,hour,depth_category,mag_category,casualties,economic_loss
0,us6000qw60,2025-07-29 23:24:52,2026-03-13 19:30:07,52.4948,160.2395,35.00,8.8,mww,"2025 Kamchatka Peninsula, Russia Earthquake",reviewed,...,Russia Earthquake,2025,7,29,Tuesday,23,shallow,great,0,0.0
1,us6000qw60,2025-07-29 23:24:52,2026-03-13 19:30:07,52.4948,160.2395,35.00,8.8,mww,"2025 Kamchatka Peninsula, Russia Earthquake",reviewed,...,Russia Earthquake,2025,7,29,Tuesday,23,shallow,great,0,0.0
2,us6000qw60,2025-07-29 23:24:52,2026-03-13 19:30:07,52.4948,160.2395,35.00,8.8,mww,"2025 Kamchatka Peninsula, Russia Earthquake",reviewed,...,Russia Earthquake,2025,7,29,Tuesday,23,shallow,great,0,0.0
3,us6000qw60,2025-07-29 23:24:52,2026-03-13 19:30:07,52.4948,160.2395,35.00,8.8,mww,"2025 Kamchatka Peninsula, Russia Earthquake",reviewed,...,Russia Earthquake,2025,7,29,Tuesday,23,shallow,great,0,0.0
4,us6000qw60,2025-07-29 23:24:52,2026-03-13 19:30:07,52.4948,160.2395,35.00,8.8,mww,"2025 Kamchatka Peninsula, Russia Earthquake",reviewed,...,Russia Earthquake,2025,7,29,Tuesday,23,shallow,great,0,0.0
5,us6000qw60,2025-07-29 23:24:52,2026-03-13 19:30:07,52.4948,160.2395,35.00,8.8,mww,"2025 Kamchatka Peninsula, Russia Earthquake",reviewed,...,Russia Earthquake,2025,7,29,Tuesday,23,shallow,great,0,0.0
6,us6000qw60,2025-07-29 23:24:52,2026-03-13 19:30:07,52.4948,160.2395,35.00,8.8,mww,"2025 Kamchatka Peninsula, Russia Earthquake",reviewed,...,Russia Earthquake,2025,7,29,Tuesday,23,shallow,great,0,0.0
7,us6000qw60,2025-07-29 23:24:52,2025-12-05 21:04:23,52.4948,160.2395,35.00,8.8,mww,"2025 Kamchatka Peninsula, Russia Earthquake",reviewed,...,Russia Earthquake,2025,7,29,Tuesday,23,shallow,great,0,0.0
8,ak0219neiszm,2021-07-29 06:15:49,2025-11-20 19:53:32,55.3635,-157.8876,35.00,8.2,mww,"2021 Chignik, Alaska Earthquake",reviewed,...,Alaska Earthquake,2021,7,29,Thursday,6,shallow,great,0,0.0
9,us7000dflf,2021-03-04 19:28:33,2025-11-26 16:39:08,-29.7228,-177.2794,28.93,8.1,mww,"2021 Kermadec Islands, New Zealand Earthquake",reviewed,...,New Zealand Earthquake,2021,3,4,Thursday,19,shallow,great,0,0.0


writing app file

In [110]:
!pip install streamlit pyngrok


In [111]:
%%writefile app.py
import streamlit as st
import pandas as pd
from sqlalchemy import create_engine

# ---------------------------------------------------------
# PAGE CONFIG
# ---------------------------------------------------------

st.set_page_config(
    page_title="Earthquake Analytics Dashboard",
    page_icon="🌍",
    layout="centered"
)

# ---------------------------------------------------------
# PROFESSIONAL UI STYLE
# ---------------------------------------------------------

st.markdown("""
<style>

.stApp{
    background: linear-gradient(180deg,#0f172a,#020617);
    color:white;
}

h1{
    text-align:center;
    font-size:48px !important;
    font-weight:800;
}

.subtitle{
    text-align:center;
    color:#cbd5e1;
    margin-bottom:30px;
}

/* dropdown center */
div[data-baseweb="select"]{
    max-width:600px;
    margin:auto;
}

/* dropdown style */
.stSelectbox div div{
    background-color:#f1f5f9;
    border-radius:10px;
}

/* center button */
div.stButton{
    display:flex;
    justify-content:center;
}

/* button style */
div.stButton > button{
    background:#1e293b;
    border:2px solid #ef4444;
    color:white;
    border-radius:10px;
    height:45px;
    width:170px;
    font-weight:bold;
}

div.stButton > button:hover{
    background:#334155;
}

</style>
""", unsafe_allow_html=True)

# ---------------------------------------------------------
# DATABASE CONNECTION
# ---------------------------------------------------------

TIDB_HOST = "gateway01.ap-southeast-1.prod.aws.tidbcloud.com"
TIDB_PORT = 4000
TIDB_USER = "31PZhzCNeELY4jm.root"
TIDB_PASSWORD = "O8N8mrUbCKtnPi7G"
TIDB_DATABASE = "earthquake"

SSL_CA_PATH = "/content/isrgrootx1 (3).pem"

engine = create_engine(
    f"mysql+pymysql://{TIDB_USER}:{TIDB_PASSWORD}@{TIDB_HOST}:{TIDB_PORT}/{TIDB_DATABASE}",
    connect_args={"ssl": {"ca": SSL_CA_PATH}},
)

def run_query(sql):
    return pd.read_sql(sql, engine)

# ---------------------------------------------------------
# QUESTIONS
# ---------------------------------------------------------

QUESTIONS = {

"1. Top 10 strongest earthquakes":
"SELECT * FROM earthquake ORDER BY mag DESC LIMIT 10;",

"2. Top 10 deepest earthquake":
"SELECT * FROM earthquake ORDER BY depth_km DESC LIMIT 10;",

"3. Shallow earthquake < 50km & mag > 7.5":
"SELECT * FROM earthquake WHERE depth_km < 50 AND mag > 7.5;",

"4. Average depth per country":
"""
SELECT country, AVG(depth_km) AS avg_depth
FROM earthquake GROUP BY country ORDER BY avg_depth DESC;
""",

"5. Average magnitude by magType":
"""
SELECT magType, AVG(mag) AS avg_mag
FROM earthquake GROUP BY magType ORDER BY avg_mag DESC;
""",

"6. Year with most earthquake":
"""
SELECT year, COUNT(*) AS total
FROM earthquake GROUP BY year
ORDER BY total DESC LIMIT 1;
""",

"7. Month with highest earthquake":
"""
SELECT year, month, COUNT(*) AS total
FROM earthquake GROUP BY year, month
ORDER BY total DESC LIMIT 1;
""",

"8. Most common day of week":
"""
SELECT day_of_week, COUNT(*) AS total
FROM earthquake GROUP BY day_of_week ORDER BY total DESC;
""",

"9. Earthquakes per hour":
"""
SELECT hour, COUNT(*) AS total
FROM earthquake GROUP BY hour ORDER BY hour;
""",

"10. Most active reporting network":
"""
SELECT net, COUNT(*) AS total
FROM earthquake GROUP BY net ORDER BY total DESC LIMIT 1;
""",

"11. Top 5 locations with highest casualties":
"""
SELECT place, SUM(casualties) AS total
FROM earthquake GROUP BY place
ORDER BY total DESC LIMIT 5;
""",

"12. Total economic loss per country":
"""
SELECT country, SUM(economic_loss) AS total
FROM earthquake GROUP BY country ORDER BY total DESC;
""",

"13. Avg economic loss by alert level":
"""
SELECT alert, AVG(economic_loss) AS avg_loss
FROM earthquake GROUP BY alert ORDER BY avg_loss DESC;
""",

"14. Reviewed vs Automatic":
"""
SELECT status, COUNT(*) AS total
FROM earthquake GROUP BY status;
""",

"15. Earthquake type count":
"""
SELECT type, COUNT(*) AS total
FROM earthquake GROUP BY type ORDER BY total DESC;
""",

"16. Data type counts":
"""
SELECT
SUM(CASE WHEN types LIKE '%dyfi%' THEN 1 END) AS dyfi_count,
SUM(CASE WHEN types LIKE '%shakemap%' THEN 1 END) AS shakemap_count,
COUNT(*) AS total_events
FROM earthquake;
""",

"17. Average RMS and gap per country":
"""
SELECT country, AVG(rms) AS avg_rms, AVG(gap) AS avg_gap
FROM earthquake GROUP BY country ORDER BY avg_rms DESC;
""",

"18. Events with nst > 50":
"SELECT * FROM earthquake WHERE nst > 50;",

"19. Tsunami events per year":
"""
SELECT year, COUNT(*) AS tsunami_events
FROM earthquake WHERE tsunami=1 GROUP BY year;
""",

"20. Earthquake per alert level":
"""
SELECT alert, COUNT(*) AS total
FROM earthquake GROUP BY alert;
""",

"21. Top 5 countries by avg magnitude (past 10 years)":
"""
SELECT country, AVG(mag) AS avg_mag, COUNT(*) AS events
FROM earthquake
WHERE year >= YEAR(CURDATE()) - 10
GROUP BY country HAVING events >= 10
ORDER BY avg_mag DESC LIMIT 5;
""",

"22. Countries with shallow & deep same month":
"""
SELECT country, year, month
FROM earthquake
GROUP BY country, year, month
HAVING SUM(CASE WHEN depth_category='shallow' THEN 1 END) > 0
AND SUM(CASE WHEN depth_category='deep' THEN 1 END) > 0;
""",

"23. Year-over-year growth":
"""
WITH y AS (
 SELECT year, COUNT(*) total FROM earthquake GROUP BY year
),
g AS (
 SELECT year, total,
 LAG(total) OVER (ORDER BY year) AS prev_total
 FROM y
)
SELECT year, total, prev_total,
((total - prev_total) / prev_total) * 100 AS growth_rate
FROM g;
""",

"24. Top 3 most active countries":
"""
SELECT country, COUNT(*) AS total, AVG(mag) AS avg_mag,
COUNT(*) * AVG(mag) AS score
FROM earthquake GROUP BY country ORDER BY score DESC LIMIT 3;
""",

"25. Avg depth near equator":
"""
SELECT country, AVG(depth_km) AS avg_depth
FROM earthquake WHERE latitude BETWEEN -5 AND 5
GROUP BY country ORDER BY avg_depth DESC;
""",

"26. Shallow to deep ratio":
"""
SELECT country,
SUM(CASE WHEN depth_category='shallow' THEN 1 END) AS shallow,
SUM(CASE WHEN depth_category='deep' THEN 1 END) AS deep
FROM earthquake GROUP BY country;
""",

"27. Magnitude difference (tsunami vs non)":
"""
SELECT
AVG(CASE WHEN tsunami=1 THEN mag END) AS tsunami_mag,
AVG(CASE WHEN tsunami=0 THEN mag END) AS non_tsunami_mag
FROM earthquake;
""",

"28. Worst reliability":
"SELECT * FROM earthquake ORDER BY rms DESC, gap DESC LIMIT 50;",

"29. Consecutive quakes within 1 hour":
"""
SELECT e1.id, e2.id, e1.country
FROM earthquake e1
JOIN earthquake e2
ON e2.time > e1.time
AND TIMESTAMPDIFF(MINUTE, e1.time, e2.time) <= 60
AND e1.country = e2.country
LIMIT 100;
""",

"30. Countries with most deep quakes":
"""
SELECT country, COUNT(*) AS deep_count
FROM earthquake WHERE depth_km > 300
GROUP BY country ORDER BY deep_count DESC LIMIT 10;
"""
}

# ---------------------------------------------------------
# UI
# ---------------------------------------------------------

st.markdown("# 🌍 Earthquake Analytics Dashboard (TiDB)")
st.markdown('<p class="subtitle">Select a question to view the results below</p>', unsafe_allow_html=True)

selected_question = st.selectbox(
    "Choose a question:",
    list(QUESTIONS.keys())
)

if st.button("Run Query"):

    try:
        df = run_query(QUESTIONS[selected_question])

        st.success("Query executed successfully!")

        st.markdown("---")
        st.subheader("📊 Query Result")

        st.dataframe(df, use_container_width=True)

    except Exception as e:
        st.error(f"❌ Error: {str(e)}")

Overwriting app.py


In [112]:
!ngrok config add-authtoken 36IeTvr7PlB5X7b8Kv3Kfw6FF9r_7DoFqQJNdrUww2ADZ5jFq


Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [113]:
from pyngrok import ngrok
public_url = ngrok.connect(8501)
public_url


<NgrokTunnel: "https://unreducible-twanda-unconsumptive.ngrok-free.dev" -> "http://localhost:8501">

In [ ]:
!streamlit run app.py --server.port 8501 --server.address 0.0.0.0





  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://136.111.184.39:8501

2026-03-15 18:32:54.315 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
